Imports


In [7]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
if not hasattr(np, "unicode_"):
    np.unicode_ = np.str_
from keras_preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences 
from sklearn.model_selection import train_test_split
import pickle

Datenhochladen und filtern


In [8]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = text.split()

    new_tokens = []

    for word in tokens:
        if word not in stop_words and len(word) > 2:
            new_tokens.append(word)
    tokens = new_tokens

    return " ".join(tokens)

In [9]:
data = pd.read_json("data/Cell_Phones_and_Accessories_5.json", lines=True)
data = data[["reviewText", "overall"]]

class_mapping = {
        1.0: 0,
        3.0: 1,
        5.0: 2,
    }

data["sentiment"] = data["overall"].map(class_mapping)

data_cleaned = data.dropna(subset=["sentiment"]).copy()
data_cleaned["sentiment"] = data_cleaned["sentiment"].astype(int)

min_class_size = data_cleaned["sentiment"].value_counts().min()

data_balanced = pd.concat(
    [
        data_cleaned[data_cleaned["sentiment"] == 0].sample(
            min_class_size, random_state=2001
        ),
        data_cleaned[data_cleaned["sentiment"] == 1].sample(
            min_class_size, random_state=2001
        ),
        data_cleaned[data_cleaned["sentiment"] == 2].sample(
            min_class_size, random_state=2001
        ),
        ]
    )

print(data_balanced["sentiment"].value_counts())

data_balanced["cleaned_text"] = data_balanced["reviewText"].apply(clean_text)

tokenizer = Tokenizer(num_words=10000,oov_token="<unk>")

tokenizer.fit_on_texts(data_balanced["cleaned_text"])

print(tokenizer.word_index)

sequences = tokenizer.texts_to_sequences(data_balanced['cleaned_text'])

sequences = tokenizer.texts_to_sequences(data_balanced['cleaned_text'])
padded_sequences = pad_sequences(sequences,maxlen=120,padding="post",truncating="post")

X = padded_sequences
y = data_balanced['sentiment'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=2001, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=2001, stratify=y_train_val)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")


sentiment
0    13279
1    13279
2    13279
Name: count, dtype: int64
{'<unk>': 1, 'phone': 2, 'case': 3, 'one': 4, 'like': 5, 'screen': 6, 'would': 7, 'use': 8, 'good': 9, 'battery': 10, 'get': 11, 'great': 12, 'iphone': 13, 'well': 14, 'product': 15, 'charge': 16, 'dont': 17, 'time': 18, 'really': 19, 'charger': 20, 'work': 21, 'back': 22, 'also': 23, 'fit': 24, 'even': 25, 'much': 26, 'got': 27, 'quality': 28, 'works': 29, 'doesnt': 30, 'charging': 31, 'little': 32, 'bought': 33, 'better': 34, 'price': 35, 'protector': 36, 'still': 37, 'nice': 38, 'buy': 39, 'used': 40, 'first': 41, 'device': 42, 'using': 43, 'two': 44, 'ive': 45, 'didnt': 46, 'usb': 47, 'put': 48, 'easy': 49, 'love': 50, 'cover': 51, 'power': 52, 'way': 53, 'could': 54, 'looks': 55, 'recommend': 56, 'plastic': 57, 'cable': 58, 'hard': 59, 'need': 60, 'new': 61, 'thing': 62, 'made': 63, 'without': 64, 'want': 65, 'make': 66, 'sound': 67, 'cheap': 68, 'look': 69, 'around': 70, 'cases': 71, 'samsung': 72, 'protection':

In [10]:
import tensorflow
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from keras.metrics import Precision, Recall, Accuracy
from keras.callbacks import EarlyStopping

model = Sequential(
        [
            Embedding(
                input_dim=10000, output_dim=128, input_length=120
            ),
            Bidirectional(LSTM(64)),
            Dropout(0.5),
            Dense(3, activation="softmax"),   
        ]
    )

model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy', Precision(), Recall()]
)

model.summary()

c:\Users\Данат\Documents\Wildau\DeepLearning\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\n--- [Starting Model Training] ---")

result = model.fit(
    X_train, y_train,
    epochs = 10,
    batch_size = 64,
    validation_data=(X_val, y_val),
    callbacks = [early_stopping],
    verbose=1
    )

model.save('sentiment_model.keras')
print("Model saved to sentiment_model.keras")

print("--- [Model Training Complete] ---")



--- [Starting Model Training] ---


NameError: name 'X_train' is not defined